In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings

# Игнорируем предупреждения для чистоты вывода
warnings.filterwarnings('ignore')

# Настройка окружения и воспроизводимости
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Загрузка и подготовка данных
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test_features.csv')

# Отбираем признаки: все колонки, кроме ID и целевой переменной
# Проверяем, что признаки есть в тестовом наборе
feature_cols = [col for col in train_df.columns
                if col not in ['patient_id', 'prediction']]

# Разделяем данные на признаки и целевую переменную
X_train = train_df[feature_cols].copy()
y_train = train_df['prediction'].dropna().copy()
X_test = test_df[feature_cols].copy()
test_ids = test_df['patient_id'].copy()

# Приводим индексы train к соответствию с y_train
X_train = X_train.loc[y_train.index]

# Предобработка
# Заполняем пропуски медианным значением
imputer = SimpleImputer(strategy='median')
X_train_i = imputer.fit_transform(X_train)
X_test_i = imputer.transform(X_test)

# Стандартизируем признаки
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_i)
X_test_scaled = scaler.transform(X_test_i)

# Определяем количество классов
num_classes = len(np.unique(y_train))

# Преобразуем данные в тензоры PyTorch
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
Y_train_t = torch.tensor(y_train.values.astype(int), dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

# Создаем DataLoader для пакетной загрузки
train_dataset = TensorDataset(X_train_t, Y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Определение модели FFN
class Clasifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(Clasifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

# Инициализация модели, функции потерь и оптимизатора
input_dim = X_train_scaled.shape[1]
model = Clasifier(input_dim=input_dim, hidden_dim=128, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()  # Подходит для многоклассовой классификации
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

print("Начинаем обучение")

# Цикл обучения
epochs = 100
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()          # Обнуляем градиенты
        outputs = model(batch_X)       # Прямой проход
        loss = criterion(outputs, batch_y)  # Вычисляем потерю
        loss.backward()                # Обратный проход
        optimizer.step()               # Обновляем веса
        epoch_loss += loss.item()

    # Выводим прогресс каждые 10 эпох
    if (epoch + 1) % 10 == 0:
        avg_loss = epoch_loss / len(train_loader)
        print(f"Эпоха {epoch+1:2d}/{epochs} | Средняя потеря: {avg_loss:.4f}")

print("Обучение завершено.")

# Генерация предсказаний
model.eval()  # Переводим модель в режим оценки
with torch.no_grad():  # Отключаем вычисление градиентов
    test_outputs = model(X_test_tensor)
    # Получаем индекс класса с максимальной вероятностью
    _, predicted = torch.max(test_outputs, dim=1)

# Преобразуем тензоры обратно в массивы
predictions = predicted.cpu().numpy()

# Формирование результата
task1_predictions = pd.DataFrame({
    'patient_id': test_ids.values,
    'prediction': predictions
})

# Сохраняем в CSV
task1_predictions.to_csv('task1_predictions.csv', index=False)
print(task1_predictions.head(21))

Начинаем обучение
Эпоха 10/100 | Средняя потеря: 0.1399
Эпоха 20/100 | Средняя потеря: 0.1002
Эпоха 30/100 | Средняя потеря: 0.0819
Эпоха 40/100 | Средняя потеря: 0.0660
Эпоха 50/100 | Средняя потеря: 0.0337
Эпоха 60/100 | Средняя потеря: 0.0348
Эпоха 70/100 | Средняя потеря: 0.0561
Эпоха 80/100 | Средняя потеря: 0.0278
Эпоха 90/100 | Средняя потеря: 0.0213
Эпоха 100/100 | Средняя потеря: 0.0296
Обучение завершено.
    patient_id  prediction
0            2           0
1            3           0
2            8           0
3           18           0
4           19           0
5           20           0
6           28           0
7           29           0
8           37           0
9           43           0
10          45           0
11          47           0
12          48           1
13          54           0
14          63           0
15          68           0
16          75           0
17          79           0
18          81           0
19          90           1
20          92